# 31 · Vector & graph — Weaviate: schema, object browsing & hybrid search

**Weaviate is the mesh's vector database — but unlike a bare index, it has a *typed schema*.**
Data lives in named **collections** (Weaviate also calls them *classes*), each with declared
properties and a declared vectorizer, and every object carries both its structured fields *and*
a vector. That combination is what lets one store answer two very different questions in the same
breath: *"show me this record"* (browse by property, like any database) and *"show me records
that mean the same thing"* (search by vector). And because keyword and vector search live side by
side, Weaviate offers **native hybrid search** — BM25 lexical scoring and vector similarity fused
into one ranked result.

That is the whole thesis of a vector store *with a schema*:

> **Browse it like a database. Search it like an embedding index. Fuse both in one query.**

### This notebook is the Weaviate UI

There is no bespoke Weaviate browser in this lab — the standalone UI that was scoped for it was
dropped, and **this notebook is its replacement**. So it deliberately does the two jobs a UI would
have done, before it does anything clever:

1. **Browse** — list every collection, open one and read its *properties* and *vectorizer* config,
   then page through real objects (with their vectors). This is the "what is even in here" view a
   UI gives you for free.
2. **Search** — the three retrieval modes the store actually supports: **vector** (`near_vector`),
   **keyword** (`bm25`), and **hybrid** (the two fused, tunable by `alpha`) — plus one **raw
   GraphQL** query, the exact API the UI would have spoken underneath.

> **Read-only, throughout.** Every call here *lists*, *fetches*, or *searches*. Nothing is inserted,
> updated, or deleted in any collection — so, like the query-wave notebooks, there is **no cleanup
> section**. We also never *assume* a property name: we open the collection's config and read one
> real object first, then build every search around the fields that are actually there.

## Setup — install the v4 client

The `weaviate-client` library is **not** in the singleuser base image (which ships `polars`, `s3fs`,
`pyarrow`, `duckdb`, `fastavro`), so we install it here. We pin **v4+** — the current client, whose
`collections` API is what every cell below uses (the old v3 `client.query` surface is gone). `polars`
— used to render result frames, exactly as in notebooks `20`–`22` — is already in the image, and
`requests` rides in as a dependency of the client (we use it for the one raw-GraphQL call).

In [1]:
%pip install -q "weaviate-client>=4"

Note: you may need to restart the kernel to use updated packages.


## Connect — HTTP **and** gRPC, env-driven

Connection is **env-driven**, the same pattern the query-wave notebooks use. The committed defaults
are the **in-cluster** service DNS; a validation run overrides the four `WEAVIATE_*` variables via
env (e.g. to a NodePort) **without editing the notebook**.

The v4 client needs **two** endpoints, not one: it does metadata and object CRUD over **HTTP**
(`8080`) and runs its high-throughput queries — `near_vector`, `bm25`, `hybrid` — over **gRPC**
(`50051`). `connect_to_custom` lets us point both at the resolved host/ports. Weaviate here is
**unmeshed and anonymous** (no auth), so there is no key to pass.

As in the query notebooks, the connection is proven by the **server version** it reports back — not
by echoing the address, which the notebook never captures.

In [2]:
import os
import contextlib
import polars as pl

# The client pulls in authlib, which writes a one-off httpx-deprecation notice straight to stderr
# at import time (it bypasses the warnings filters). Silence just that import; a genuine ImportError
# would still raise, since redirect_stderr only affects writes, not exception propagation.
with open(os.devnull, "w") as _null, contextlib.redirect_stderr(_null):
    import weaviate
    from weaviate.classes.query import MetadataQuery

# committed defaults = in-cluster DNS; a validation run overrides *_HOST/*_PORT via env
client = weaviate.connect_to_custom(
    http_host=os.environ.get("WEAVIATE_HTTP_HOST", "weaviate.weyland.svc.cluster.local"),
    http_port=int(os.environ.get("WEAVIATE_HTTP_PORT", "8080")),
    http_secure=False,                       # unmeshed, plain HTTP
    grpc_host=os.environ.get("WEAVIATE_GRPC_HOST", "weaviate.weyland.svc.cluster.local"),
    grpc_port=int(os.environ.get("WEAVIATE_GRPC_PORT", "50051")),
    grpc_secure=False,
    skip_init_checks=False,                  # verify BOTH HTTP and gRPC are reachable at connect time
)

# small helper: rows + column names -> a polars DataFrame for display (house style, as in 20–22)
def frame(rows, cols):
    return pl.DataFrame(rows, schema=list(cols), orient="row")

print("ready        :", client.is_ready())                 # both transports answered
print("Weaviate ver :", client.get_meta()["version"])      # proves the connection, not the address

ready        : True
Weaviate ver : 1.38.0-rc.1


## Browse 1 — what collections exist?

The first thing a UI shows is the list of collections. `client.collections.list_all()` returns every
class's config; we pair each with its object **count** (a cheap `aggregate.over_all(total_count=True)`)
so the browse view answers *"what is in here, and how much of it"* in one frame.

Two families show up: the **`Weyland*`** classes (the platform's own RAG corpus — documentation and
code, chunked and embedded) and the **`DatasetsMusic*` / `DatasetsHealth*`** classes (the public
datasets mirrored into the mesh's vector tier).

In [3]:
cfgs = client.collections.list_all(simple=False)   # {name: CollectionConfig}
rows = []
for name in sorted(cfgs):
    coll = client.collections.get(name)
    n = coll.aggregate.over_all(total_count=True).total_count
    vzr = cfgs[name].vectorizer                      # how vectors are produced for this class
    rows.append((name, n, len(cfgs[name].properties), str(vzr)))

print(f"{len(cfgs)} collections in this Weaviate")
frame(rows, ["collection", "objects", "n_props", "vectorizer"]).sort("objects", descending=True)

12 collections in this Weaviate


collection,objects,n_props,vectorizer
str,i64,i64,str
"""DatasetsMusicUciYearPrediction""",515345,2,"""Vectorizers.NONE"""
"""DatasetsHealthOpenFoodFacts""",195792,4,"""Vectorizers.NONE"""
"""DatasetsMusicSpotifyTracks""",114000,4,"""Vectorizers.NONE"""
"""DatasetsMusicFmaFeatures""",106574,1,"""Vectorizers.NONE"""
"""DatasetsMusicAudioset""",35824,1,"""Vectorizers.NONE"""
…,…,…,…
"""DatasetsMusicFmaEchonest""",14511,3,"""Vectorizers.NONE"""
"""WeylandChunk""",7903,4,"""Vectorizers.NONE"""
"""DatasetsMusicLpMusiccapsMc""",5521,3,"""Vectorizers.NONE"""


Note the **vectorizer** column. Most classes read `NONE` — which does *not* mean "no vectors", it
means **bring-your-own-vector**: the embeddings were computed by an external model (the mesh's own
embedding pipeline) and written in alongside each object, rather than Weaviate calling out to an
inference module at ingest. That is the important schema fact to carry into search: we supply the
query vector ourselves (there is no server-side text-to-vector step to lean on), which is exactly
why the vector-search cell below **seeds its query from an existing object's own vector**.

## Browse 2 — open one collection and read its schema

We focus on **`WeylandChunk`**: the platform's documentation and code, split into chunks and
embedded — the corpus behind RAG retrieval. It is the richest text class here, which makes it the
one where keyword and hybrid search have something real to match.

`collection.config.get()` is the UI's "schema" panel: the declared **properties** with their data
types, and the **vectorizer**. Reading it *before* querying is the whole discipline of this notebook
— the search cells use only the property names printed here, never guessed ones.

In [4]:
CLASS = "WeylandChunk"
chunks = client.collections.get(CLASS)
cfg = chunks.config.get()

print(f"collection : {CLASS}")
print(f"vectorizer : {cfg.vectorizer}   (NONE => vectors are supplied at ingest, not computed here)")
print(f"objects    : {chunks.aggregate.over_all(total_count=True).total_count}")
print("\nproperties (the fields every object carries):")
frame([(p.name, str(p.data_type)) for p in cfg.properties], ["property", "data_type"])

collection : WeylandChunk
vectorizer : Vectorizers.NONE   (NONE => vectors are supplied at ingest, not computed here)
objects    : 7903

properties (the fields every object carries):


property,data_type
str,str
"""source_path""","""DataType.TEXT"""
"""chunk_index""","""DataType.INT"""
"""chunk_title""","""DataType.TEXT"""
"""content""","""DataType.TEXT"""


## Browse 3 — page through real objects (with their vectors)

This is the object-inspection view: `fetch_objects` pages straight through the store with **no
search at all** — the equivalent of scrolling the UI's object table. Passing `include_vector=True`
pulls each object's embedding too, so we can see the real property values *and* confirm the vector
dimensionality that vector search will operate in.

We render the structured properties as a frame and report the vector length separately (768 floats
per object is a lot to put in a table cell). That vector dimension is the fact the next cell depends
on: a `near_vector` query must supply a vector of exactly this width.

In [5]:
page = chunks.query.fetch_objects(limit=5, include_vector=True)

def vlen(o):
    v = o.vector.get("default") if isinstance(o.vector, dict) else o.vector
    return len(v) if v else 0

rows = [
    (
        o.properties.get("source_path", "")[:44],
        (o.properties.get("chunk_title") or "")[:36],
        o.properties.get("chunk_index"),
        (o.properties.get("content") or "").replace("\n", " ")[:60],
        vlen(o),
    )
    for o in page.objects
]
print(f"vector dimension: {vlen(page.objects[0])} floats per object")
frame(rows, ["source_path", "chunk_title", "chunk_index", "content (head)", "vec_dim"])

vector dimension: 768 floats per object


source_path,chunk_title,chunk_index,content (head),vec_dim
str,str,i64,str,i64
"""docs/arch.md""","""13. Roadmap & maintenance""",13,"""## 13. Roadmap & maintenance …",768
"""nodes/mother/lab/weyland-platf…","""""",1,""": chartName: Genre acous…",768
"""aidlc-kb/engineering-knowledge…","""What It Is""",1,"""## What It Is A versioning con…",768
"""docs/demos/vector-stores.md""","""2. Quality checks (all three s…",2,"""## 2. Quality checks (all thre…",768
"""nodes/mother/lab/weyland-platf…","""""",2,""" the context lacks the answer.…",768


## Search 1 — vector similarity (`near_vector`)

Vector search asks *"which objects mean the same thing as this one?"* Because `WeylandChunk` is a
bring-your-own-vector class (vectorizer `NONE`, from the browse above), there is no server-side
text-to-vector step — so we **seed the query with a real object's own vector**: grab one chunk with
its embedding, then ask Weaviate for its nearest neighbours.

`return_metadata=MetadataQuery(distance=True)` surfaces the **distance** for each hit — smaller is
closer. The seed object is its own nearest neighbour at distance `0.0`, which is the sanity check
that the search is working; the rows after it are the genuinely-similar chunks.

In [6]:
seed = chunks.query.fetch_objects(limit=1, include_vector=True).objects[0]
seed_vec = seed.vector["default"]
print("seed chunk :", repr((seed.properties.get("chunk_title") or seed.properties.get("source_path"))[:60]))
print("query dim  :", len(seed_vec))

near = chunks.query.near_vector(
    near_vector=seed_vec,
    limit=5,
    return_metadata=MetadataQuery(distance=True),
    return_properties=["chunk_title", "source_path"],
)
rows = [
    (round(o.metadata.distance, 4),
     (o.properties.get("chunk_title") or "")[:40],
     (o.properties.get("source_path") or "")[:48])
    for o in near.objects
]
frame(rows, ["distance", "chunk_title", "source_path"])

seed chunk : '13. Roadmap & maintenance'
query dim  : 768


distance,chunk_title,source_path
f64,str,str
0.0,"""13. Roadmap & maintenance""","""docs/arch.md"""
0.1623,"""UI walkthrough (eyes-on)""","""docs/demos/port-iac-coverage.m…"
0.1811,"""""","""nodes/mother/lab/weyland-platf…"
0.1826,"""Outstanding (before any row is…","""docs/demos/README.md"""
0.1948,"""""","""docs/backlog.md"""


## Search 2 — keyword scoring (`bm25`)

The other half of the store is **lexical**. `bm25` ignores vectors entirely and ranks objects by
the classic BM25 term-frequency score over their text properties — the same family of ranking a
search engine uses. It finds the objects that literally contain the query *terms*, which is exactly
what vector search can miss (a rare identifier, an exact keyword) and exactly what a UI's search box
does first.

We search a term we know is in the corpus from the browse above; `MetadataQuery(score=True)` returns
the BM25 score (higher is a stronger lexical match).

In [7]:
TERM = "dagster"   # a real term from the corpus (platform docs mention it heavily)
kw = chunks.query.bm25(
    query=TERM,
    limit=5,
    return_metadata=MetadataQuery(score=True),
    return_properties=["chunk_title", "source_path"],
)
print(f"bm25 '{TERM}': {len(kw.objects)} hits")
rows = [
    (round(o.metadata.score, 4),
     (o.properties.get("chunk_title") or "")[:40],
     (o.properties.get("source_path") or "")[:48])
    for o in kw.objects
]
frame(rows, ["bm25_score", "chunk_title", "source_path"])

bm25 'dagster': 5 hits


bm25_score,chunk_title,source_path
f64,str,str
3.7526,"""Dagster""","""docs/validation/test-commands.…"
3.4853,"""""","""nodes/mother/lab/weyland-platf…"
3.4216,"""""","""nodes/mother/lab/weyland-platf…"
3.3261,"""""","""nodes/mother/lab/weyland-platf…"
3.2991,"""""","""nodes/mother/lab/weyland-platf…"


## Search 3 — hybrid (BM25 **+** vector, fused)

This is Weaviate's headline capability: **one query that runs both searches and fuses their
rankings.** `hybrid` takes both a `query` string (scored by BM25) and a `vector` (scored by
similarity), and blends the two result lists by **`alpha`**:

> **`alpha = 0.0` is pure BM25 · `alpha = 1.0` is pure vector · `alpha = 0.5` weights them equally.**

Hybrid is what you reach for when you want *both* the exact-keyword recall of BM25 *and* the
semantic reach of vectors — the query term pins the literal matches while the vector pulls in
conceptually-related chunks that share no words. We reuse the seed vector from the vector-search
cell and the term from the keyword cell, at `alpha=0.5` so neither dominates. The returned `score`
is the fused ranking score.

In [8]:
hy = chunks.query.hybrid(
    query=TERM,
    vector=seed_vec,
    alpha=0.5,                 # 0 = all BM25, 1 = all vector, 0.5 = balanced
    limit=5,
    return_metadata=MetadataQuery(score=True),
    return_properties=["chunk_title", "source_path"],
)
print(f"hybrid (alpha=0.5) of vector(seed) + bm25('{TERM}'): {len(hy.objects)} hits")
rows = [
    (round(o.metadata.score, 4),
     (o.properties.get("chunk_title") or "")[:40],
     (o.properties.get("source_path") or "")[:48])
    for o in hy.objects
]
frame(rows, ["hybrid_score", "chunk_title", "source_path"])

hybrid (alpha=0.5) of vector(seed) + bm25('dagster'): 5 hits


hybrid_score,chunk_title,source_path
f64,str,str
0.5,"""Dagster""","""docs/validation/test-commands.…"
0.5,"""13. Roadmap & maintenance""","""docs/arch.md"""
0.4243,"""""","""nodes/mother/lab/weyland-platf…"
0.4237,"""""","""nodes/mother/lab/weyland-platf…"
0.4055,"""""","""nodes/mother/lab/weyland-platf…"


## The raw API — one GraphQL query

Everything above went through the Python client's typed `collections` methods. Underneath, Weaviate
speaks **GraphQL** — and that is the exact API a browser UI would have called. To show it plainly we
POST one query straight to the `/v1/graphql` REST endpoint (built from the same env-driven host/port
as the client, so nothing is hard-coded) and read the JSON back.

The query mirrors the BM25 search: `Get { WeylandChunk(bm25: {...}) { … _additional { id score } } }`.
The `_additional` block is Weaviate's channel for search metadata — the object `id` and the match
`score` — alongside the requested properties. This is the shape every client, typed or not, is
sugar over.

In [9]:
import requests   # rides in as a dependency of weaviate-client

# build the endpoint from the SAME env-driven parts the client used (no address is hard-coded)
gql_url = "http://{h}:{p}/v1/graphql".format(
    h=os.environ.get("WEAVIATE_HTTP_HOST", "weaviate.weyland.svc.cluster.local"),
    p=os.environ.get("WEAVIATE_HTTP_PORT", "8080"),
)
gql = {
    "query": '''{
      Get {
        WeylandChunk(limit: 3, bm25: {query: "%s"}) {
          chunk_title
          source_path
          _additional { id score }
        }
      }
    }''' % TERM
}
resp = requests.post(gql_url, json=gql, timeout=30)
resp.raise_for_status()
hits = resp.json()["data"]["Get"]["WeylandChunk"]
print(f"GraphQL returned {len(hits)} objects")
rows = [
    (round(float(h["_additional"]["score"]), 4),
     (h.get("chunk_title") or "")[:40],
     (h.get("source_path") or "")[:48])
    for h in hits
]
frame(rows, ["bm25_score", "chunk_title", "source_path"])

GraphQL returned 3 objects


bm25_score,chunk_title,source_path
f64,str,str
3.7526,"""Dagster""","""docs/validation/test-commands.…"
3.4853,"""""","""nodes/mother/lab/weyland-platf…"
3.3261,"""""","""nodes/mother/lab/weyland-platf…"


## Close the client

The v4 client holds an HTTP session and a gRPC channel open; closing them is the one bit of tidy-up
this read-only notebook owes. (There is nothing to clean up in the *store* — we created nothing.)

In [10]:
client.close()
print("client closed")

client closed


## When to reach for Weaviate

Weaviate is the mesh's store for **"find me things that mean this"**, and its schema is what makes
it more than a raw index:

- **Semantic retrieval / RAG** — the core job. `near_vector` over an embedded corpus (the
  `WeylandChunk` documentation+code above) is exactly what a retrieval step in a RAG pipeline does.
- **Hybrid search when keywords *and* meaning both matter** — `hybrid` with a tuned `alpha` is the
  one call that gives you BM25's exact-term recall and vector recall together. Reach for it when a
  pure-vector search keeps missing a literal identifier, or a pure-keyword search keeps missing a
  paraphrase.
- **Browse + inspect a typed object store** — because collections have declared properties, you can
  page and filter objects like a database, not just probe an index. That is the job this notebook
  stands in for now that the bespoke UI is gone.

**Reach elsewhere when the shape is different:**

| you want to… | use |
|--------------|-----|
| semantic / hybrid search over embedded text, with a schema to browse | **Weaviate** (this notebook) |
| a pure vector index tuned for raw ANN throughput, minimal schema | **Qdrant** (the mesh's other vector store) |
| relationship / graph traversal ("what connects to what") | **Neo4j** (the graph tier) |
| federated SQL across the lakehouse + operational DBs | **Trino** (notebook `20`) |
| a fast point read/write on one specialist store | **native client** (notebook `22`) |

> **Browse the schema, then search it three ways.** Weaviate earns its place in the vector/graph
> wave by being the store you can both *read like a catalogue* and *query by meaning* — and hybrid
> search is the reason you keep BM25 and vectors in the same system instead of two.